# WUSTL-IIoT-2021: Binary Intrusion Detection (Coursework)

**Dataset:** Zolanvari et al. — network flow features from an Industrial IoT testbed (~1.19M flows).  
**Task:** Classify **normal** (0) vs **attack** (1). Severe class imbalance (~7.3% attacks).

**Protocol (authors):** Remove leakage columns before training so the model generalises. Scaling lives **inside** `sklearn.pipeline.Pipeline` to avoid leakage across CV folds.

**Reproducibility:** Fixed random seeds for splits, CV, and estimators that support `random_state`.

## 1. Configuration and imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    f1_score,
    matthews_corrcoef,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

# --- Paths: dataset at repository root (wustl_iiot_2021.csv) ---
def _repo_root(marker: str = "wustl_iiot_2021.csv") -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / marker).is_file():
            return d
    raise FileNotFoundError(f"Place {marker} at the repository root (searched from {here}).")

DATA_PATH = _repo_root() / "wustl_iiot_2021.csv"

# Columns that leak attack identity / must be removed per dataset documentation
LEAKAGE_COLS = ["StartTime", "LastTime", "SrcAddr", "DstAddr", "sIpId", "dIpId"]

RANDOM_STATE = 42
# Stratified subset for hyperparameter search & fast CV (full data is ~1.2M rows)
TUNE_SAMPLE_SIZE = 150_000
# Optional cap when fitting the final model on training data (None = use all X_train)
FINAL_TRAIN_CAP = 500_000

sns.set_theme(style="whitegrid", context="notebook")

## 2. Load data, remove leakage columns, build labels

We keep **`Traffic`** only for EDA / optional multiclass checks; modelling uses binary **`y`** from **`Target`** (verify against `Traffic`).

In [ ]:
print(f"Loading: {DATA_PATH}")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(df.shape)
print(df.columns.tolist())

missing = [c for c in LEAKAGE_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Expected leakage columns not found: {missing}")

df_model = df.drop(columns=LEAKAGE_COLS).copy()
# Features: all numeric / usable columns except label columns
feature_cols = [c for c in df_model.columns if c not in ("Traffic", "Target")]
X = df_model[feature_cols].copy()
# Ensure numeric features for sklearn (encode any stray object columns)
for col in X.select_dtypes(include=["object", "category"]).columns:
    X[col], _ = X[col].factorize()
X = X.apply(pd.to_numeric, errors="coerce")
if X.isna().any().any():
    X = X.fillna(0)
y = df_model["Target"].astype(int)

# Sanity: binary label matches Traffic
y_from_traffic = (df_model["Traffic"].str.lower() != "normal").astype(int)
assert (y.values == y_from_traffic.values).all(), "Target inconsistent with Traffic"
print("Label check OK: Target matches (Traffic != 'normal').")

del df  # free memory if possible
X.info()

## 3. Exploratory analysis (class balance & traffic mix)

In [ ]:
vc = y.value_counts().sort_index()
print("Binary class counts:\n", vc)
print("Attack rate: {:.4f}".format(y.mean()))

traffic_counts = df_model["Traffic"].value_counts()
print("\nTraffic type counts (top 10):\n", traffic_counts.head(10))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
vc.plot(kind="bar", ax=ax[0], color=["steelblue", "coral"])
ax[0].set_title("Binary label (Target)")
ax[0].set_xticklabels(["normal (0)", "attack (1)"], rotation=0)
traffic_counts.head(8).plot(kind="barh", ax=ax[1], color="teal")
ax[1].set_title("Top traffic types (raw)")
plt.tight_layout()
plt.show()

# Correlation heatmap on a random stratified sample (full matrix is expensive)
sample_n = min(12_000, len(X))
rng = np.random.default_rng(RANDOM_STATE)
idx = rng.choice(np.arange(len(X)), size=sample_n, replace=False)
X_s = X.iloc[idx]
# numeric only
num = X_s.select_dtypes(include=[np.number])
if num.shape[1] >= 2:
    c = num.corr().abs()
    plt.figure(figsize=(14, 12))
    sns.heatmap(c, cmap="viridis", square=False)
    plt.title(f"Feature |correlation| (sample n={sample_n})")
    plt.tight_layout()
    plt.show()

## 4. Stratified hold-out split

**Test set is used once** for final reporting. All tuning uses **train** only (via CV on a stratified tuning subset).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)
print("Train:", X_train.shape, "Attack rate:", y_train.mean())
print("Test :", X_test.shape, "Attack rate:", y_test.mean())

# Stratified tuning subset from TRAIN only
if len(X_train) > TUNE_SAMPLE_SIZE:
    X_tune, _, y_tune, _ = train_test_split(
        X_train,
        y_train,
        train_size=TUNE_SAMPLE_SIZE,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )
else:
    X_tune, y_tune = X_train, y_train
print(f"\nTuning subset: {X_tune.shape[0]} rows (stratified from train)")

## 5. Metrics (imbalanced IDS)

For rare attacks, **accuracy is misleading**. We report:

- **PR-AUC** (`average_precision`) — primary for imbalance  
- **F1** (attack = positive class)  
- **MCC** — correlation between prediction and truth under imbalance  
- **ROC-AUC** — supplementary (can be optimistic when negatives dominate)

`StratifiedKFold` keeps attack prevalence stable in each fold.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 6. Baselines: Logistic Regression, Gaussian NB, kNN

All use **`Pipeline([StandardScaler, clf])`** so scaling is learned on training data only (including inside CV).

**Note:** k-NN training scales poorly; we **fit kNN** on a **stratified cap** (`KNN_TRAIN_CAP`) while other models use `FINAL_TRAIN_CAP`.

In [ ]:
KNN_TRAIN_CAP = 80_000


def stratified_cap(X_df, y_s, max_rows, seed=RANDOM_STATE):
    if len(X_df) <= max_rows:
        return X_df, y_s
    X_sub, _, y_sub, _ = train_test_split(
        X_df,
        y_s,
        train_size=max_rows,
        stratify=y_s,
        random_state=seed,
    )
    return X_sub, y_sub


pipe_lr = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "clf",
            LogisticRegression(
                class_weight="balanced",
                solver="saga",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

pipe_nb = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", GaussianNB()),
    ]
)

pipe_knn = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15, n_jobs=-1)),
    ]
)

baseline_results = []
for name, est in [
    ("LogisticRegression", pipe_lr),
    ("GaussianNB", pipe_nb),
    ("kNN (k=15)", pipe_knn),
]:
    if name.startswith("kNN"):
        X_cv, y_cv = stratified_cap(X_tune, y_tune, KNN_TRAIN_CAP)
    else:
        X_cv, y_cv = X_tune, y_tune
    out = cross_validate(
        est,
        X_cv,
        y_cv,
        cv=CV,
        scoring={"ap": "average_precision", "f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=-1,
    )
    baseline_results.append(
        {
            "model": name,
            "pr_auc_mean": out["test_ap"].mean(),
            "pr_auc_std": out["test_ap"].std(),
            "f1_mean": out["test_f1"].mean(),
            "roc_auc_mean": out["test_roc_auc"].mean(),
        }
    )

pd.DataFrame(baseline_results).round(4)

## 7. Tree models: Decision Tree & Random Forest (+ light tuning)

Random Forest matches the dataset authors’ baseline narrative; we tune **`n_estimators`** and **`max_depth`** on the tuning subset only.

In [ ]:
pipe_dt = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "clf",
            DecisionTreeClassifier(
                class_weight="balanced",
                max_depth=24,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

pipe_rf = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "clf",
            RandomForestClassifier(
                class_weight="balanced_subsample",
                n_estimators=200,
                max_depth=None,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

param_dist = {
    "clf__n_estimators": [100, 200, 400],
    "clf__max_depth": [None, 24, 32],
    "clf__min_samples_leaf": [1, 2, 4],
}

search_rf = RandomizedSearchCV(
    pipe_rf,
    param_distributions=param_dist,
    n_iter=12,
    cv=CV,
    scoring="average_precision",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
search_rf.fit(X_tune, y_tune)
print("Best RF params:", search_rf.best_params_)
print("Best CV PR-AUC:", search_rf.best_score_)

best_rf = search_rf.best_estimator_

tree_rows = []
for name, est in [("DecisionTree", pipe_dt), ("RandomForest (tuned)", best_rf)]:
    out = cross_validate(
        est,
        X_tune,
        y_tune,
        cv=CV,
        scoring={"ap": "average_precision", "f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=-1,
    )
    tree_rows.append(
        {
            "model": name,
            "pr_auc_mean": out["test_ap"].mean(),
            "f1_mean": out["test_f1"].mean(),
            "roc_auc_mean": out["test_roc_auc"].mean(),
        }
    )
pd.DataFrame(tree_rows).round(4)

## 8. Linear SVM (linear kernel)

Fast linear margin on scaled features; uses **`decision_function`** scores (no probabilities unless calibrated).

In [ ]:
pipe_svc = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "clf",
            LinearSVC(
                class_weight="balanced",
                dual=False,
                max_iter=3000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

out_svc = cross_validate(
    pipe_svc,
    X_tune,
    y_tune,
    cv=CV,
    scoring={"ap": "average_precision", "f1": "f1", "roc_auc": "roc_auc"},
    n_jobs=-1,
)
print(
    "LinearSVC CV PR-AUC: {:.4f} ± {:.4f}".format(
        out_svc["test_ap"].mean(),
        out_svc["test_ap"].std(),
    )
)

## 9. Soft voting ensemble (probability-based)

We combine **Logistic Regression**, **tuned Random Forest**, and **Gaussian NB** (all provide `predict_proba`, trained on the **same** capped training set).

In [ ]:
from sklearn.base import clone

ensemble = VotingClassifier(
    estimators=[
        ("lr", clone(pipe_lr)),
        ("rf", clone(search_rf.best_estimator_)),
        ("nb", clone(pipe_nb)),
    ],
    voting="soft",
    n_jobs=-1,
)

out_ens = cross_validate(
    ensemble,
    X_tune,
    y_tune,
    cv=CV,
    scoring={"ap": "average_precision", "f1": "f1", "roc_auc": "roc_auc"},
    n_jobs=-1,
)
print(
    "Voting (LR+RF+NB) CV PR-AUC: {:.4f} ± {:.4f}".format(
        out_ens["test_ap"].mean(),
        out_ens["test_ap"].std(),
    )
)

## 10. Final evaluation on the held-out **test** set

Models are **re-fit** on a capped training set (`FINAL_TRAIN_CAP`). kNN uses **`KNN_TRAIN_CAP`**. Metrics use **attack as the positive class**.

In [ ]:
X_fit, y_fit = stratified_cap(X_train, y_train, FINAL_TRAIN_CAP)
X_knn_fit, y_knn_fit = stratified_cap(X_train, y_train, KNN_TRAIN_CAP)


def eval_on_test(name, model, X_tr, y_tr):
    m = clone(model)
    m.fit(X_tr, y_tr)
    if hasattr(m, "predict_proba"):
        scores = m.predict_proba(X_test)[:, 1]
    else:
        scores = m.decision_function(X_test)
    y_pred = m.predict(X_test)
    return {
        "model": name,
        "pr_auc": average_precision_score(y_test, scores),
        "roc_auc": roc_auc_score(y_test, scores),
        "f1_attack": f1_score(y_test, y_pred, pos_label=1),
        "mcc": matthews_corrcoef(y_test, y_pred),
    }


rows = []
rows.append(eval_on_test("LogisticRegression", pipe_lr, X_fit, y_fit))
rows.append(eval_on_test("GaussianNB", pipe_nb, X_fit, y_fit))
rows.append(eval_on_test("kNN", pipe_knn, X_knn_fit, y_knn_fit))
rows.append(eval_on_test("DecisionTree", pipe_dt, X_fit, y_fit))
rows.append(eval_on_test("RandomForest (tuned)", clone(search_rf.best_estimator_), X_fit, y_fit))
rows.append(eval_on_test("LinearSVC", pipe_svc, X_fit, y_fit))
rows.append(eval_on_test("Voting LR+RF+NB", clone(ensemble), X_fit, y_fit))

summary = pd.DataFrame(rows).sort_values("pr_auc", ascending=False).reset_index(drop=True)
summary.round(4)

### 10b. PR / ROC curves and confusion matrix (best test PR-AUC model)

In [ ]:
model_specs = {
    "LogisticRegression": (clone(pipe_lr), X_fit, y_fit),
    "GaussianNB": (clone(pipe_nb), X_fit, y_fit),
    "kNN": (clone(pipe_knn), X_knn_fit, y_knn_fit),
    "DecisionTree": (clone(pipe_dt), X_fit, y_fit),
    "RandomForest (tuned)": (clone(search_rf.best_estimator_), X_fit, y_fit),
    "LinearSVC": (clone(pipe_svc), X_fit, y_fit),
    "Voting LR+RF+NB": (clone(ensemble), X_fit, y_fit),
}

best_name = summary.iloc[0]["model"]
champ_m, Xtr, ytr = model_specs[best_name]
champ_m.fit(Xtr, ytr)

if hasattr(champ_m, "predict_proba"):
    s_test = champ_m.predict_proba(X_test)[:, 1]
else:
    s_test = champ_m.decision_function(X_test)
y_hat = champ_m.predict(X_test)

precision, recall, thresh = precision_recall_curve(y_test, s_test)
fpr, tpr, _ = roc_curve(y_test, s_test)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(recall, precision, label=f"PR-AUC={average_precision_score(y_test, s_test):.4f}")
ax[0].set_xlabel("Recall (attack)")
ax[0].set_ylabel("Precision")
ax[0].set_title(f"Precision–Recall — {best_name}")
ax[0].legend()

ax[1].plot(fpr, tpr, label=f"ROC-AUC={roc_auc_score(y_test, s_test):.4f}")
ax[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
ax[1].set_xlabel("FPR")
ax[1].set_ylabel("TPR")
ax[1].set_title(f"ROC — {best_name}")
ax[1].legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_hat,
    display_labels=["normal", "attack"],
    normalize="true",
    ax=ax,
    colorbar=False,
)
ax.set_title(f"Normalised confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_hat, target_names=["normal", "attack"]))

## 11. Permutation importance (Random Forest)

We measure how much **test-set** AP drops when a feature is shuffled (sklearn `permutation_importance`). Computed on a **row subset** of the test set for speed.

In [ ]:
rf_perm = clone(search_rf.best_estimator_)
rf_perm.fit(X_fit, y_fit)

n_perm = min(25_000, len(X_test))
rng = np.random.default_rng(RANDOM_STATE + 1)
perm_idx = rng.choice(np.arange(len(X_test)), size=n_perm, replace=False)
X_perm = X_test.iloc[perm_idx]
y_perm = y_test.iloc[perm_idx]

pi = permutation_importance(
    rf_perm,
    X_perm,
    y_perm,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1,
)

imp = pd.Series(pi.importances_mean, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(10, 8))
imp.head(20).sort_values().plot(kind="barh", color="steelblue")
plt.title("Top 20 permutation importances (RF, AP score)")
plt.tight_layout()
plt.show()
imp.head(15).round(5)

## 12. Limitations & ethics (brief)

- **Sampling caps** (`TUNE_SAMPLE_SIZE`, `FINAL_TRAIN_CAP`, kNN cap) trade runtime against using every training row; document when you change them.
- **Stationarity:** Model is fit on one testbed window; real plants drift (firmware, workloads). Deployment needs monitoring and periodic retraining.
- **Adversarial ML:** Attackers may craft flows to evade a static classifier; this coursework evaluates **statistical** detection, not robustness to adaptive adversaries.
- **Operational costs:** High recall on attacks may inflate false alarms for operators; thresholds should reflect organisational tolerance (not only default 0.5).